# 03 - KITTI Preprocessing

This notebook prepares the KITTI dataset for model training by:
- validating raw image/label/calibration folders
- parsing annotation files
- creating a train split if needed
- summarizing category distribution and sample counts

This is the bridge between dataset audit and model training.

In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd().resolve().parent
DATA_ROOT = ROOT / 'data' / 'KITTI'
RAW = DATA_ROOT / 'raw'
SPLITS = DATA_ROOT / 'splits'

required_dirs = [RAW / 'image_2', RAW / 'label_2', RAW / 'calib']
for d in required_dirs:
    print(f'{d.name}:', 'OK' if d.exists() else 'MISSING')

SPLITS.mkdir(exist_ok=True, parents=True)
print('split dir:', SPLITS)

In [ ]:
# Build a simple train split from image filenames if missing
image_dir = RAW / 'image_2'
train_file = SPLITS / 'train.txt'

if image_dir.exists():
    ids = sorted(p.stem for p in image_dir.glob('*.png'))
    if ids:
        if not train_file.exists() or train_file.read_text(encoding='utf-8').strip() != '\n'.join(ids):
            train_file.write_text('\n'.join(ids) + '\n', encoding='utf-8')
            print(f'Wrote {len(ids)} sample IDs to {train_file}')
        else:
            print(f'{train_file} already exists with {len(ids)} IDs')
    else:
        print('No PNG files found in image_2.')
else:
    print('image_2 directory is missing. Please download the full KITTI dataset first.')

In [ ]:
# Quick annotation summary
label_dir = RAW / 'label_2'
if label_dir.exists():
    files = sorted(label_dir.glob('*.txt'))
    category_counts = {}
    for f in files[:100]:
        with f.open('r', encoding='utf-8') as fh:
            for line in fh:
                parts = line.strip().split()
                if len(parts) < 16:
                    continue
                cls = parts[0]
                category_counts[cls] = category_counts.get(cls, 0) + 1

    print('annotation files inspected:', min(len(files), 100))
    print('category counts:', dict(sorted(category_counts.items())))
else:
    print('label_2 directory is missing.')

In [ ]:
# Example of parsing a KITTI label line
sample = RAW / 'label_2'
if sample.exists():
    files = sorted(sample.glob('*.txt'))
    if files:
        with files[0].open('r', encoding='utf-8') as fh:
            for line in fh:
                print(line.strip())
                break
    else:
        print('No label files found.')
else:
    print('label_2 not found.')

## Preprocessing checklist

Before training, verify all of the following are true:
- raw images exist in `image_2`
- label annotations exist in `label_2`
- calibration files exist in `calib`
- a `splits/train.txt` file has been created

Once those checks pass, the next stage is training and evaluation.